In [ ]:
import os
import csv
import pytesseract
from PIL import Image, ImageOps, ImageEnhance
from transformers import pipeline
from pdf2image import convert_from_path
from pathlib import Path

# --- SYSTEM CONFIGURATION ---
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
os.environ['TESSDATA_PREFIX'] = r'C:\Program Files\Tesseract-OCR\tessdata'
POPPLER_PATH = r'C:\Program Files\poppler\Library\bin'

TESSERACT_CONFIG = {"config": "--psm 3 --oem 3"}
CONFIDENCE_THRESHOLD = 0.35  # Galat data filter karne ke liye threshold

# --- PATHS SETUP ---
INPUT_DIR = r"File Path"
OUTPUT_CSV = os.path.join(INPUT_DIR, "final_outpu.csv")

# --- LOAD MODEL (No Token Needed - 100% Public & Active) ---
print("Loading Document QA Pipeline...")
nlp_invoice = pipeline(
    "document-question-answering", 
    model="impira/layoutlm-document-qa", 
    tesseract_kwargs=TESSERACT_CONFIG
)

# --- TARGET QUESTIONS (Synonyms list) ---
QUESTIONS = {
    "account_number": ["What is the ACCOUNT NUMBER?", "Account No:", "Account #"],
    "customer_name": ["What is the CUSTOMER NAME?", "Name of consumer:", "Bill to name:"],
    "service_address": ["What is the SERVICE ADDRESS?", "Location address:", "Property Address:"],
    "billing_date": ["What is the BILLING DATE?", "Statement Date:", "Date of issue:"],
    "due_date": ["What is the DUE DATE?", "Payment due date:", "Pay by date:"],
    "service_period": ["What is the SERVICE PERIOD?", "Billing period:", "Service dates:"],
    "previous_balance": ["What is the PREVIOUS BALANCE?", "Balance forward:"],
    "payments_credited": ["What is the amount for PAYMENTS?", "Payments received:", "Credits:"],
    "current_cycle_charges": ["What is the TOTAL CURRENT CHARGES?", "New charges total:"],
    "net_amount_due": ["What is the TOTAL AMOUNT DUE?", "Net amount due:", "Total amount payable:"],
    "electric_subtotal": ["What is the Electric Service Total?", "Electricity charges:"],
    "gas_subtotal": ["What is the Gas Service Total?", "Gas charges:"],
    "sewer_subtotal": ["What is the charge for SEWER SERVICE?", "Sewer total:"]
}

# --- MAIN PROCESSING PIPELINE ---
if not os.path.exists(INPUT_DIR):
    print(f"Error: Path '{INPUT_DIR}' nahi mila. Path check karein.")
else:
    all_rows = []
    fieldnames = ["File_Name"] + list(QUESTIONS.keys())

    # Files loop
    for file_name in os.listdir(INPUT_DIR):
        file_path = os.path.join(INPUT_DIR, file_name)

        # Basic validation (skipping folders and empty files)
        if os.path.isdir(file_path) or os.path.getsize(file_path) == 0:
            continue

        # Extract pages based on file type
        pages = []
        if file_name.lower().endswith(".pdf"):
            try:
                pages = convert_from_path(Path(file_path), poppler_path=POPPLER_PATH)
            except Exception as e:
                print(f"PDF convert karne mein error ({file_name}): {e}")
                continue
        else:
            try:
                img = Image.open(file_path)
                img = ImageOps.grayscale(img)
                img = ImageEnhance.Contrast(img).enhance(2.0)
                if img.width < 1500:
                    img = img.resize((img.width * 2, img.height * 2), Image.Resampling.LANCZOS)
                pages.append(img)
            except Exception as e:
                print(f"Image open karne mein error ({file_name}): {e}")
                continue

        # Process each page
        for index, page_image in enumerate(pages):
            print(f"\n>>> Processing: {file_name} | Page: {index + 1}")
            
            row_data = {field: "" for field in fieldnames}
            row_data["File_Name"] = file_name

            # Ask each question
            for key, queries in QUESTIONS.items():
                best_answer = ""
                best_score = 0.0
                
                for q in queries:
                    try:
                        result = nlp_invoice(image=page_image, question=q)
                        if isinstance(result, list):
                            result = result[0]
                        
                        if result['score'] > best_score:
                            best_score = result['score']
                            best_answer = result['answer']
                    except Exception as e:
                        continue

                # Filter and clean output
                if best_score >= CONFIDENCE_THRESHOLD:
                    clean_text = best_answer.strip().replace('\n', ' ')
                    
                    # Date Validation Guardrail
                    if key in ["billing_date", "due_date"] and not any(char.isdigit() for char in clean_text):
                        row_data[key] = ""
                    else:
                        print(f"   {key}: {clean_text} ({best_score:.2f})")
                        row_data[key] = clean_text
                else:
                    row_data[key] = ""
            
            all_rows.append(row_data)

    # --- SAVE RESULTS TO CSV ---
    with open(OUTPUT_CSV, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)
        
    print(f"\n[SUCCESS] Extraction mukammal! Data save ho chuka hai yahan:\n{OUTPUT_CSV}")

Loading Document QA Pipeline...


config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

e:\conda\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SaniteX\.cache\huggingface\hub\models--impira--layoutlm-document-qa. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/511M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


>>> Processing: 1.jpg | Page: 1
   account_number: 1234567890-00002 (1.00)
   customer_name: JOHN (0.77)
   service_address: 123 MAIN ST (0.98)
   billing_date: 01/05/2023 (1.00)
   due_date: 01/05/2023 (0.97)
   service_period: 01/05/2023 (1.00)
   previous_balance: $220.35 (0.99)
   payments_credited: we-energies.com (0.39)
   current_cycle_charges: $253.78 (1.00)
   net_amount_due: $186.68 (0.40)
   electric_subtotal: $186.68 (1.00)
   gas_subtotal: $67.10 (1.00)
   sewer_subtotal: $186.68 (0.97)

>>> Processing: basic_residential_page_1.jpg | Page: 1
   account_number: 00000-00000 (1.00)
   customer_name: CUSTAMER (1.00)
   service_address: 1-800-642-4272 (0.68)
   billing_date: Aug 17, 2025 (0.95)
   due_date: Aug 17, 2025 (1.00)
   service_period: Jul 24, 2025 (0.99)
   previous_balance: 97.5 (1.00)
   payments_credited: $ 101.56 (1.00)
   current_cycle_charges: $ 101.56 (0.43)
   net_amount_due: $ 101.56 (1.00)
   electric_subtotal: 62.49 (1.00)
   gas_subtotal: 39.07 (1.00)
  